In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
import time
import logging
import os

warnings.filterwarnings('ignore')

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ==================== 1. EXACT DFE BACKBONE FROM FIGURE 2 ====================
class DFEBackbone(nn.Module):
    def __init__(self, input_channels=1, embedding_dim=256):
        super(DFEBackbone, self).__init__()
        
        # EXACTLY as in Figure 2 of the paper
        # Total: 5 conv64, 4 conv128, 4 conv256 = 13 conv layers
        
        # Block 1: f1 (2 conv64)
        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 64, kernel_size=3, padding=1, stride=2)  # stride=2
        self.bn2 = nn.BatchNorm1d(64)
        
        # Block 2: f2 (2 conv64)
        self.conv3 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(64)
        self.conv4 = nn.Conv1d(64, 64, kernel_size=3, padding=1, stride=2)  # stride=2
        self.bn4 = nn.BatchNorm1d(64)
        
        # Block 3: f3 (1 conv64) - فقط ۱ لایه conv64 اینجا
        self.conv5 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm1d(64)
        self.conv6 = nn.Conv1d(64, 64, kernel_size=3, padding=1, stride=2)  # stride=2
        self.bn6 = nn.BatchNorm1d(64)
        
        # Block 4: f4 (2 conv128)
        self.conv7 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn7 = nn.BatchNorm1d(128)
        self.conv8 = nn.Conv1d(128, 128, kernel_size=3, padding=1, stride=2)  # stride=2
        self.bn8 = nn.BatchNorm1d(128)
        
        # Block 5: f5 (2 conv128)
        self.conv9 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.bn9 = nn.BatchNorm1d(128)
        self.conv10 = nn.Conv1d(128, 128, kernel_size=3, padding=1)  # no stride
        self.bn10 = nn.BatchNorm1d(128)
        
        # Block 6: f6 (4 conv256)
        self.conv11 = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.bn11 = nn.BatchNorm1d(256)
        self.conv12 = nn.Conv1d(256, 256, kernel_size=3, padding=1)  # no stride
        self.bn12 = nn.BatchNorm1d(256)
        self.conv13 = nn.Conv1d(256, 256, kernel_size=3, padding=1)  # no stride
        self.bn13 = nn.BatchNorm1d(256)
        self.conv14 = nn.Conv1d(256, 256, kernel_size=3, padding=1)  # no stride
        self.bn14 = nn.BatchNorm1d(256)
        
        # Adaptive average pooling
        self.adaptive_pool = nn.AdaptiveAvgPool1d(1)
        
        # Fully connected layers: fc 256 (as in paper Fig. 2)
        self.fc1 = nn.Linear(256, 256)
        self.fc2 = nn.Linear(256, embedding_dim)
        
        self.relu = nn.ReLU()
        
    def forward(self, x, return_features=False):
        """Forward pass with feature extraction f1-f6"""
        features = []
        
        # Block 1: f1
        x1 = self.relu(self.bn1(self.conv1(x)))
        x1 = self.relu(self.bn2(self.conv2(x1)))  # stride=2
        features.append(x1)
        
        # Block 2: f2
        x2 = self.relu(self.bn3(self.conv3(x1)))
        x2 = self.relu(self.bn4(self.conv4(x2)))  # stride=2
        features.append(x2)
        
        # Block 3: f3
        x3 = self.relu(self.bn5(self.conv5(x2)))
        x3 = self.relu(self.bn6(self.conv6(x3)))  # stride=2
        features.append(x3)
        
        # Block 4: f4
        x4 = self.relu(self.bn7(self.conv7(x3)))
        x4 = self.relu(self.bn8(self.conv8(x4)))  # stride=2
        features.append(x4)
        
        # Block 5: f5
        x5 = self.relu(self.bn9(self.conv9(x4)))
        x5 = self.relu(self.bn10(self.conv10(x5)))  # no stride
        features.append(x5)
        
        # Block 6: f6 (4 conv256 layers)
        x6 = self.relu(self.bn11(self.conv11(x5)))
        x6 = self.relu(self.bn12(self.conv12(x6)))
        x6 = self.relu(self.bn13(self.conv13(x6)))
        x6 = self.relu(self.bn14(self.conv14(x6)))
        features.append(x6)
        
        # Adaptive pooling
        x_pool = self.adaptive_pool(x6)
        x_pool = x_pool.view(x_pool.size(0), -1)
        
        # Fully connected layers
        embedding = self.relu(self.fc1(x_pool))
        embedding = self.fc2(embedding)
        
        if return_features:
            return embedding, features
        return embedding

# ==================== 2. EXACT PAPER IMPLEMENTATION OF FEATURE COMPRESSOR ====================
class FeatureCompressor(nn.Module):
    def __init__(self, beta_values=[0.01, 0.02, 0.03, 0.04, 0.05, 0.05]):
        super(FeatureCompressor, self).__init__()
        self.register_buffer('beta', torch.tensor(beta_values))
        
    def gaussian_kld(self, mu_p, cov_p, mu_q, cov_q):
        """
        Equation 7 from paper: KLD between two Gaussian distributions
        D[N(μ_p, Λ_p) || N(μ_q, Λ_q)]
        """
        d = mu_p.shape[0]
        
        # Add regularization for numerical stability
        eps = 1e-6
        cov_p_reg = cov_p + eps * torch.eye(d, device=mu_p.device)
        cov_q_reg = cov_q + eps * torch.eye(d, device=mu_q.device)
        
        # Compute determinants
        det_cov_p = torch.det(cov_p_reg)
        det_cov_q = torch.det(cov_q_reg)
        
        # Compute inverse of covariance q
        cov_q_inv = torch.inverse(cov_q_reg)
        
        # Compute trace term
        trace_term = torch.trace(cov_q_inv @ cov_p_reg)
        
        # Compute quadratic term
        diff = mu_q - mu_p
        quad_term = diff.T @ cov_q_inv @ diff
        
        # KLD formula (Equation 7)
        kld = 0.5 * (torch.log(det_cov_q / det_cov_p) + quad_term + trace_term - d)
        
        return kld
    
    def compute_mutual_info_bound(self, f_i, f_j):
        """
        Equation 6 from paper: Non-parametric upper bound for mutual information
        I(f_i; f_j) ≤ Ï(f_i; f_j) = -1/N Σ_p log(1/N Σ_q exp(-D[N(μ_i^p, Λ_i^p) || N(μ_i^q, Λ_i^q)]))
        """
        N = f_i.shape[0]  # batch size
        
        # Reshape features: each sample is treated as a Gaussian component
        # f_i shape: (N, C, L) -> (N, D) where D = C * L
        f_i_flat = f_i.view(N, -1)
        f_j_flat = f_j.view(N, -1)
        
        # For each sample, treat it as mean of a Gaussian
        # Assume identity covariance for simplicity (as mentioned in paper assumptions)
        mus_i = f_i_flat  # Shape: (N, D)
        mus_j = f_j_flat  # Shape: (N, D)
        
        # Create identity covariance matrices for each sample
        D = mus_i.shape[1]
        identity = torch.eye(D, device=f_i.device)
        
        # Compute pairwise KLD between all samples
        mutual_info_bound = 0.0
        
        for p in range(N):
            sum_exp = 0.0
            for q in range(N):
                # KLD between Gaussian p and Gaussian q (both with identity covariance)
                # For N(μ_p, I) and N(μ_q, I), KLD = 0.5 * ||μ_p - μ_q||^2
                kld = 0.5 * torch.sum((mus_i[p] - mus_i[q]) ** 2)
                sum_exp += torch.exp(-kld)
            
            # Avoid log(0)
            sum_exp = torch.clamp(sum_exp, min=1e-8)
            mutual_info_bound += torch.log(sum_exp / N)
        
        mutual_info_bound = -mutual_info_bound / N
        
        return mutual_info_bound
    
    def compute_compression_loss(self, features):
        """
        Equation 8 from paper: Feature compression loss upper bound
        L_f ≤ Σ_{i=0}^{5} β_i * Ï(f_i; f_{i+1})
        """
        total_loss = torch.tensor(0.0, device=features[0].device)
        
        for i in range(len(features) - 1):
            f_i = features[i]
            f_j = features[i + 1]
            
            # Compute mutual information upper bound
            I_bound = self.compute_mutual_info_bound(f_i, f_j)
            
            # Apply beta weight (Equation 8)
            if i < len(self.beta):
                total_loss = total_loss + self.beta[i] * I_bound
        
        return total_loss

# ==================== 3. MODIFIED TRIPLET LOSS FROM PAPER ====================
class ModifiedTripletLoss(nn.Module):
    def __init__(self, alpha1=0.5, alpha2=2.0):
        super(ModifiedTripletLoss, self).__init__()
        self.alpha1 = alpha1
        self.alpha2 = alpha2
        
    def forward(self, anchor, positive, negative):
        """Equation 12 from paper"""
        d_pos = F.pairwise_distance(anchor, positive, p=2)
        d_neg = F.pairwise_distance(anchor, negative, p=2)
        
        # Original triplet component
        triplet_loss = F.relu(d_pos - d_neg + self.alpha1)
        
        # Intra-class constraint (Equation 11)
        intra_class_loss = F.relu(d_pos - self.alpha2)
        
        # Combined loss (Equation 12)
        loss = triplet_loss + 0.5 * intra_class_loss
        
        return loss.mean()

# ==================== 4. COMPLETE DFE MODEL ====================
class DFEComplete(nn.Module):
    def __init__(self, input_channels=1, embedding_dim=256):
        super(DFEComplete, self).__init__()
        self.backbone = DFEBackbone(input_channels, embedding_dim)
        self.feature_compressor = FeatureCompressor()
        
    def forward(self, x, return_features=False):
        if return_features:
            embedding, features = self.backbone(x, return_features=True)
            return embedding, features
        else:
            embedding = self.backbone(x)
            return embedding

# ==================== 5. DFE TRAINING FRAMEWORK ====================
class DFE:
    def __init__(self, embedding_dim=256, alpha1=0.5, alpha2=2.0, 
                 batch_size=32, learning_rate=0.001, epochs=200):
        
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        logger.info(f"Using device: {self.device}")
        
        # Paper parameters
        self.embedding_dim = embedding_dim
        self.alpha1 = alpha1
        self.alpha2 = alpha2
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.epochs = epochs
        
        self.model = None
        self.template_library = {}
        self.label_encoder = LabelEncoder()
        self.scaler = StandardScaler()
        self.beta_values = [0.01, 0.02, 0.03, 0.04, 0.05, 0.05]
    
    def prepare_data(self, csv_path, is_training=True):
        """Load and preprocess data"""
        logger.info(f"Loading {csv_path}")
        df = pd.read_csv(csv_path)
        
        X = df.iloc[:, :-1].values
        y = df.iloc[:, -1].values
        
        logger.info(f"Dataset shape: {X.shape}, Classes: {len(np.unique(y))}")
        
        if is_training:
            y_encoded = self.label_encoder.fit_transform(y)
            X_scaled = self.scaler.fit_transform(X)
        else:
            y_encoded = self.label_encoder.transform(y)
            X_scaled = self.scaler.transform(X)
        
        return X_scaled, y_encoded
    
    def create_triplet_dataset(self, X, y):
        """Create triplets for training"""
        triplets = []
        
        unique_classes = np.unique(y)
        class_indices = {cls: np.where(y == cls)[0] for cls in unique_classes}
        
        for cls in unique_classes:
            cls_idx = class_indices[cls]
            other_idx = np.concatenate([v for k, v in class_indices.items() if k != cls])
            
            if len(cls_idx) > 0 and len(other_idx) > 0:
                for i in range(min(len(cls_idx), 100)):  # Limit triplets per class
                    anchor_idx = cls_idx[i % len(cls_idx)]
                    anchor = X[anchor_idx]
                    
                    # Positive
                    pos_idx = cls_idx[np.random.randint(0, len(cls_idx))]
                    while pos_idx == anchor_idx and len(cls_idx) > 1:
                        pos_idx = cls_idx[np.random.randint(0, len(cls_idx))]
                    positive = X[pos_idx]
                    
                    # Negative
                    neg_idx = other_idx[np.random.randint(0, len(other_idx))]
                    negative = X[neg_idx]
                    
                    triplets.append([anchor, positive, negative])
        
        logger.info(f"Created {len(triplets)} triplets")
        return np.array(triplets)
    
    def train(self, train_csv_path):
        """Train DFE model exactly as in paper"""
        logger.info("="*60)
        logger.info("STARTING DFE TRAINING (Exact Paper Implementation)")
        logger.info("="*60)
        
        # Load data
        X_train, y_train = self.prepare_data(train_csv_path, is_training=True)
        
        # Create triplets
        triplets = self.create_triplet_dataset(X_train, y_train)
        
        # Initialize model with EXACT architecture
        self.model = DFEComplete(embedding_dim=self.embedding_dim).to(self.device)
        
        # Count parameters
        total_params = sum(p.numel() for p in self.model.parameters())
        logger.info(f"Model parameters: {total_params:,}")
        logger.info(f"Architecture: 5 conv64, 4 conv128, 4 conv256 (13 conv layers total)")
        
        # Loss functions
        triplet_loss_fn = ModifiedTripletLoss(alpha1=self.alpha1, alpha2=self.alpha2)
        feature_compressor = FeatureCompressor(beta_values=self.beta_values)
        
        # Optimizer: SGD with constant LR as in paper
        optimizer = torch.optim.SGD(self.model.parameters(), lr=self.learning_rate, momentum=0.9)
        
        # Convert to tensor (keep on CPU for now to avoid CUDA issues)
        triplets_tensor = torch.FloatTensor(triplets)
        
        logger.info(f"Training for {self.epochs} epochs")
        logger.info(f"Batch size: {self.batch_size}")
        logger.info(f"Learning rate: {self.learning_rate}")
        logger.info(f"α1={self.alpha1}, α2={self.alpha2}")
        logger.info(f"β={self.beta_values}")
        
        best_loss = float('inf')
        
        for epoch in range(self.epochs):
            self.model.train()
            epoch_loss = 0
            epoch_triplet_loss = 0
            epoch_compression_loss = 0
            
            # Shuffle indices on CPU
            indices = torch.randperm(len(triplets_tensor))
            
            num_batches = 0
            for i in range(0, len(indices), self.batch_size):
                batch_indices = indices[i:i + self.batch_size]
                if len(batch_indices) < 2:  # Need at least 2 for batch norm
                    continue
                
                # Get batch (still on CPU)
                batch_triplets = triplets_tensor[batch_indices]
                
                # Move to device
                anchors = batch_triplets[:, 0].unsqueeze(1).to(self.device)
                positives = batch_triplets[:, 1].unsqueeze(1).to(self.device)
                negatives = batch_triplets[:, 2].unsqueeze(1).to(self.device)
                
                optimizer.zero_grad()
                
                try:
                    # Forward pass with features for compression
                    anchor_emb, anchor_features = self.model(anchors, return_features=True)
                    positive_emb, _ = self.model(positives, return_features=True)
                    negative_emb = self.model(negatives)
                    
                    # Triplet loss
                    triplet_loss = triplet_loss_fn(anchor_emb, positive_emb, negative_emb)
                    
                    # Feature compression loss (include input as f0)
                    input_as_feature = anchors.view(anchors.shape[0], -1).unsqueeze(-1)
                    all_features = [input_as_feature] + anchor_features
                    compression_loss = feature_compressor.compute_compression_loss(all_features)
                    
                    # Total loss
                    loss = triplet_loss + compression_loss
                    
                    # Backward pass
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                    optimizer.step()
                    
                    epoch_loss += loss.item()
                    epoch_triplet_loss += triplet_loss.item()
                    epoch_compression_loss += compression_loss.item() if compression_loss > 0 else 0
                    num_batches += 1
                    
                except Exception as e:
                    logger.warning(f"Error in batch {i//self.batch_size}: {e}")
                    continue
            
            if num_batches > 0:
                avg_loss = epoch_loss / num_batches
                avg_triplet = epoch_triplet_loss / num_batches
                avg_compression = epoch_compression_loss / num_batches
                
                # Log progress
                logger.info(f"Epoch [{epoch+1:3d}/{self.epochs}] "
                          f"Loss: {avg_loss:.4f} "
                          f"(Triplet: {avg_triplet:.4f}, "
                          f"Comp: {avg_compression:.4f})")
                
                # Save best model
                if avg_loss < best_loss:
                    best_loss = avg_loss
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': self.model.state_dict(),
                        'loss': avg_loss,
                    }, 'dfe_best_model.pth')
            
            # Early stopping check
            if epoch > 50 and avg_loss > best_loss * 1.5:
                logger.info(f"Early stopping at epoch {epoch+1}")
                break
        
        logger.info("Training completed!")
        
        # Load best model
        if os.path.exists('dfe_best_model.pth'):
            checkpoint = torch.load('dfe_best_model.pth', map_location=self.device)
            self.model.load_state_dict(checkpoint['model_state_dict'])
            logger.info(f"Loaded best model from epoch {checkpoint['epoch']+1}")
        
        # Build template library with 20 samples per class
        self._build_template_library(X_train, y_train)
    
    def _build_template_library(self, X, y):
        """Build template library with 20 samples per class as in paper"""
        logger.info("Building template library...")
        self.model.eval()
        
        unique_classes = np.unique(y)
        
        with torch.no_grad():
            for cls in unique_classes:
                class_indices = np.where(y == cls)[0]
                
                # Take up to 20 samples per class
                n_samples = min(20, len(class_indices))
                selected_indices = np.random.choice(class_indices, n_samples, replace=False)
                
                # Get embeddings
                class_samples = X[selected_indices]
                class_samples_tensor = torch.FloatTensor(class_samples).unsqueeze(1).to(self.device)
                embeddings = self.model(class_samples_tensor)
                
                # Average embedding as template
                template = torch.mean(embeddings, dim=0).cpu().numpy()
                self.template_library[cls] = template
        
        logger.info(f"Template library created with {len(self.template_library)} classes")
    
    def evaluate(self, test_csv_path):
        """Evaluate model"""
        logger.info("="*60)
        logger.info("EVALUATION")
        logger.info("="*60)
        
        X_test, y_test = self.prepare_data(test_csv_path, is_training=False)
        
        # Predict
        y_pred = self._predict_batch(X_test)
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        
        logger.info(f"Accuracy:  {accuracy:.4f}")
        logger.info(f"Precision: {precision:.4f}")
        logger.info(f"Recall:    {recall:.4f}")
        logger.info(f"F1-Score:  {f1:.4f}")
        
        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        }
    
    def _predict_batch(self, X, batch_size=32):
        """Predict in batches to avoid memory issues"""
        self.model.eval()
        predictions = []
        
        with torch.no_grad():
            for i in range(0, len(X), batch_size):
                batch_X = X[i:i + batch_size]
                X_tensor = torch.FloatTensor(batch_X).unsqueeze(1).to(self.device)
                embeddings = self.model(X_tensor).cpu().numpy()
                
                for emb in embeddings:
                    min_dist = float('inf')
                    pred_cls = -1
                    
                    for cls, template in self.template_library.items():
                        dist = np.linalg.norm(emb - template)
                        if dist < min_dist:
                            min_dist = dist
                            pred_cls = cls
                    
                    predictions.append(pred_cls)
        
        return np.array(predictions)

# ==================== 6. MAIN ====================
def main():
    print("="*60)
    print("DEEP FLOW EMBEDDING (DFE) - Exact Paper Implementation")
    print("="*60)
    
    # Initialize DFE with paper parameters
    dfe = DFE(
        embedding_dim=256,
        alpha1=0.5,      # From paper page 8
        alpha2=2.0,      # From paper page 8
        batch_size=32,   # From paper page 8
        learning_rate=0.001,  # From paper page 8
        epochs=145       # From paper page 8
    )
    
    # Use your CSV files
    train_csv = "../DATASETS/CDR-MLC/level_1.csv"
    test_csv = "../DATASETS/CDR-MLC/level_3.csv"
    
    # Train
    dfe.train(train_csv)
    
    # Evaluate
    results = dfe.evaluate(test_csv)
    
    # Save final model
    torch.save({
        'model_state_dict': dfe.model.state_dict(),
        'template_library': dfe.template_library,
        'label_encoder': dfe.label_encoder,
        'scaler': dfe.scaler
    }, 'dfe_final_model.pth')
    
    print("\nModel saved to 'dfe_final_model.pth'")
    print(f"\nFinal Results: Accuracy={results['accuracy']:.4f}, F1={results['f1_score']:.4f}")

if __name__ == "__main__":
    main()

2026-02-08 08:17:48,916 - INFO - Using device: cuda
2026-02-08 08:17:48,917 - INFO - ============================================================
2026-02-08 08:17:48,917 - INFO - STARTING DFE TRAINING (Exact Paper Implementation)
2026-02-08 08:17:48,917 - INFO - ============================================================
2026-02-08 08:17:48,917 - INFO - Loading ../DATASETS/CDR-MLC/level_1.csv


DEEP FLOW EMBEDDING (DFE) - Exact Paper Implementation


2026-02-08 08:17:49,127 - INFO - Dataset shape: (68079, 37), Classes: 5
2026-02-08 08:17:49,166 - INFO - Created 500 triplets
2026-02-08 08:17:49,206 - INFO - Model parameters: 1,059,136
2026-02-08 08:17:49,207 - INFO - Architecture: 5 conv64, 4 conv128, 4 conv256 (13 conv layers total)
2026-02-08 08:17:49,223 - INFO - Training for 145 epochs
2026-02-08 08:17:49,223 - INFO - Batch size: 32
2026-02-08 08:17:49,223 - INFO - Learning rate: 0.001
2026-02-08 08:17:49,223 - INFO - α1=0.5, α2=2.0
2026-02-08 08:17:49,224 - INFO - β=[0.01, 0.02, 0.03, 0.04, 0.05, 0.05]
2026-02-08 08:18:13,026 - INFO - Epoch [  1/145] Loss: 1.3054 (Triplet: 0.6586, Comp: 0.6468)
2026-02-08 08:18:34,588 - INFO - Epoch [  2/145] Loss: 1.2430 (Triplet: 0.5936, Comp: 0.6494)
2026-02-08 08:18:55,892 - INFO - Epoch [  3/145] Loss: 1.1728 (Triplet: 0.5238, Comp: 0.6490)
2026-02-08 08:19:17,237 - INFO - Epoch [  4/145] Loss: 1.1240 (Triplet: 0.4798, Comp: 0.6442)
2026-02-08 08:19:38,549 - INFO - Epoch [  5/145] Loss: 1.


Model saved to 'dfe_final_model.pth'

Final Results: Accuracy=0.0810, F1=0.0373


## Dynamic

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
import warnings
import time
import logging
import os
from torch.utils.data import Dataset, DataLoader
import math

warnings.filterwarnings('ignore')

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ==================== 1. EXACT DFE BACKBONE WITH DYNAMIC DIMENSIONS ====================
class DFEBackbone(nn.Module):
    def __init__(self, input_channels=1, embedding_dim=256, input_features=81):
        super(DFEBackbone, self).__init__()
        
        # Calculate input dimensions (find closest square)
        self.input_features = input_features
        self.input_size = self._calculate_input_size(input_features)
        logger.info(f"Input features: {input_features}, Input size: {self.input_size}x{self.input_size}")
        
        # Block 1: f1 (2 conv64 layers) - بدون downsampling
        self.conv1 = nn.Conv2d(input_channels, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1, stride=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        # Block 2: f2 (2 conv64 layers)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1, stride=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1, stride=1)
        self.bn4 = nn.BatchNorm2d(64)  # تصحیح: تغییر از bn به bn4
        
        # Block 3: f3 (1 conv64 layer)
        self.conv5 = nn.Conv2d(64, 64, kernel_size=3, padding=1, stride=1)
        self.bn5 = nn.BatchNorm2d(64)
        
        # Block 4: f4 (2 conv128 layers)
        self.conv6 = nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=1)
        self.bn6 = nn.BatchNorm2d(128)
        self.conv7 = nn.Conv2d(128, 128, kernel_size=3, padding=1, stride=1)
        self.bn7 = nn.BatchNorm2d(128)
        
        # Block 5: f5 (2 conv128 layers)  
        self.conv8 = nn.Conv2d(128, 128, kernel_size=3, padding=1, stride=1)
        self.bn8 = nn.BatchNorm2d(128)
        self.conv9 = nn.Conv2d(128, 128, kernel_size=3, padding=1, stride=1)
        self.bn9 = nn.BatchNorm2d(128)
        
        # Block 6: f6 (4 conv256 layers)
        self.conv10 = nn.Conv2d(128, 256, kernel_size=3, padding=1, stride=1)
        self.bn10 = nn.BatchNorm2d(256)
        self.conv11 = nn.Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        self.bn11 = nn.BatchNorm2d(256)
        self.conv12 = nn.Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        self.bn12 = nn.BatchNorm2d(256)
        self.conv13 = nn.Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        self.bn13 = nn.BatchNorm2d(256)
        
        # Adaptive average pooling
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Fully connected layers
        self.fc1 = nn.Linear(256, 256)
        self.fc2 = nn.Linear(256, embedding_dim)
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        
        # Residual connections
        self.downsample1 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=1, stride=1),
            nn.BatchNorm2d(64)
        )
        
    def _calculate_input_size(self, num_features):
        """Calculate the closest square size for given number of features"""
        # Find the square root
        sqrt = math.isqrt(num_features)
        
        # Check if it's a perfect square
        if sqrt * sqrt == num_features:
            return sqrt
        
        # Otherwise find the closest square that can contain all features
        for size in range(sqrt + 1, 0, -1):
            if size * size >= num_features:
                return size
        
        return sqrt + 1  # Fallback
    
    def forward(self, x, return_features=False):
        """Forward pass با ابعاد پویا"""
        features = []
        
        # Block 1: f1
        x1 = self.relu(self.bn1(self.conv1(x)))  # (batch, 64, H, W)
        identity = x1
        x1 = self.relu(self.bn2(self.conv2(x1)))  # (batch, 64, H, W)
        identity = self.downsample1(identity)  # (batch, 64, H, W)
        x1 = x1 + identity
        features.append(x1.view(x1.size(0), -1))
        
        # Block 2: f2
        identity = x1
        x2 = self.relu(self.bn3(self.conv3(x1)))
        x2 = self.relu(self.bn4(self.conv4(x2)))  # تصحیح: استفاده از bn4
        x2 = x2 + identity
        features.append(x2.view(x2.size(0), -1))
        
        # Block 3: f3
        identity = x2
        x3 = self.relu(self.bn5(self.conv5(x2)))
        x3 = x3 + identity
        features.append(x3.view(x3.size(0), -1))
        
        # Block 4: f4
        x4 = self.relu(self.bn6(self.conv6(x3)))  # (batch, 128, H, W)
        identity = x4
        x4 = self.relu(self.bn7(self.conv7(x4)))
        x4 = x4 + identity
        features.append(x4.view(x4.size(0), -1))
        
        # Block 5: f5
        identity = x4
        x5 = self.relu(self.bn8(self.conv8(x4)))
        x5 = self.relu(self.bn9(self.conv9(x5)))
        x5 = x5 + identity
        features.append(x5.view(x5.size(0), -1))
        
        # Block 6: f6
        x6 = self.relu(self.bn10(self.conv10(x5)))  # (batch, 256, H, W)
        identity = x6
        x6 = self.relu(self.bn11(self.conv11(x6)))
        x6 = self.relu(self.bn12(self.conv12(x6)))
        x6 = self.relu(self.bn13(self.conv13(x6)))
        x6 = x6 + identity
        features.append(x6.view(x6.size(0), -1))
        
        # Adaptive pooling
        x_pool = self.adaptive_pool(x6)  # (batch, 256, 1, 1)
        x_pool = x_pool.view(x_pool.size(0), -1)  # (batch, 256)
        
        # Fully connected layers
        embedding = self.relu(self.fc1(x_pool))
        embedding = self.dropout(embedding)
        embedding = self.fc2(embedding)  # (batch, embedding_dim)
        
        if return_features:
            all_features = [x.view(x.size(0), -1)] + features
            return embedding, all_features
        return embedding

# ==================== 2. EXACT PAPER FEATURE COMPRESSOR (SIMPLIFIED) ====================
class FeatureCompressor(nn.Module):
    def __init__(self, beta_values=[0.01, 0.02, 0.03, 0.04, 0.05, 0.05]):
        super(FeatureCompressor, self).__init__()
        self.register_buffer('beta', torch.tensor(beta_values))
        
    def compute_mutual_info_bound(self, f_i, f_j):
        """
        اصلاح شده: هندل کردن featureهای با ابعاد مختلف
        """
        # Ensure both features have the same batch size
        batch_size = min(f_i.size(0), f_j.size(0))
        f_i = f_i[:batch_size]
        f_j = f_j[:batch_size]
        
        # اگر ابعاد featureها متفاوت است، projection انجام دهید
        if f_i.size(1) != f_j.size(1):
            # Project به بعد مشترک (مثلاً 256)
            target_dim = min(f_i.size(1), f_j.size(1), 256)
            
            if f_i.size(1) > target_dim:
                # کاهش بعد f_i
                f_i_proj = nn.Linear(f_i.size(1), target_dim, bias=False).to(f_i.device)
                f_i = f_i_proj(f_i)
            
            if f_j.size(1) > target_dim:
                # کاهش بعد f_j
                f_j_proj = nn.Linear(f_j.size(1), target_dim, bias=False).to(f_j.device)
                f_j = f_j_proj(f_j)
        
        # Normalize
        f_i_norm = F.normalize(f_i, p=2, dim=1)
        f_j_norm = F.normalize(f_j, p=2, dim=1)
        
        # Compute similarity (حالا ابعاد مطابقت دارند)
        similarity_matrix = torch.mm(f_i_norm, f_j_norm.t())
        
        # Upper bound estimation
        mean_similarity = torch.mean(torch.diag(similarity_matrix))
        mutual_info_bound = -torch.log(torch.clamp(mean_similarity, min=1e-8))
        
        return mutual_info_bound
    
    def compute_compression_loss(self, features):
        """
        Equation 8 from paper: Feature compression loss upper bound
        """
        total_loss = torch.tensor(0.0, device=features[0].device)
        
        for i in range(len(features) - 1):
            f_i = features[i]
            f_j = features[i + 1]
            
            # Compute mutual information upper bound
            I_bound = self.compute_mutual_info_bound(f_i, f_j)
            
            # Apply beta weight
            if i < len(self.beta):
                total_loss = total_loss + self.beta[i] * I_bound
        
        return total_loss

# ==================== 3. EXACT MODIFIED TRIPLET LOSS FROM PAPER ====================
class ModifiedTripletLoss(nn.Module):
    def __init__(self, alpha1=0.5, alpha2=2.0, eta=0.5):
        super(ModifiedTripletLoss, self).__init__()
        self.alpha1 = alpha1  # margin for inter-class constraint
        self.alpha2 = alpha2  # margin for intra-class constraint  
        self.eta = eta        # weight coefficient η in Equation 12
        
    def forward(self, anchor, positive, negative):
        """
        Exact implementation of Equation 12 from paper:
        L_NTC = 1/N Σ max{δ_w((x_a,x_p)|(x_a,x_n)) + α1, 0} + η * 1/N Σ max{d(φ_w(x_a), φ_w(x_p)), α2}
        """
        # Distance calculations
        d_pos = F.pairwise_distance(anchor, positive, p=2)  # d(φ_w(x_a), φ_w(x_p))
        d_neg = F.pairwise_distance(anchor, negative, p=2)  # d(φ_w(x_a), φ_w(x_n))
        
        # Inter-class constraint: δ_w((x_a,x_p)|(x_a,x_n)) = d_pos - d_neg
        inter_class_loss = F.relu(d_pos - d_neg + self.alpha1)
        
        # Intra-class constraint: d(φ_w(x_a), φ_w(x_p)) ≤ α2  
        intra_class_loss = F.relu(d_pos - self.alpha2)
        
        # Combined loss (Equation 12)
        total_loss = inter_class_loss.mean() + self.eta * intra_class_loss.mean()
        
        return total_loss

# ==================== 4. TRIPLET DATASET ====================
class TripletDataset(Dataset):
    def __init__(self, X, y, samples_per_class=100, input_size=None):
        self.input_size = input_size
        self.triplets = self._generate_triplets(X, y, samples_per_class)
        
    def _generate_triplets(self, X, y, samples_per_class):
        """Generate triplets for training"""
        triplets = []
        unique_classes = np.unique(y)
        
        # Create class index mapping
        class_indices = {}
        for cls in unique_classes:
            class_indices[cls] = np.where(y == cls)[0]
        
        for cls in unique_classes:
            cls_indices = class_indices[cls]
            other_classes = [c for c in unique_classes if c != cls]
            
            if len(cls_indices) < 2 or len(other_classes) == 0:
                continue
                
            # Generate triplets for this class
            n_triplets = min(samples_per_class, len(cls_indices))
            
            for i in range(n_triplets):
                # Anchor
                anchor_idx = cls_indices[i % len(cls_indices)]
                anchor = X[anchor_idx]
                
                # Positive (same class, different sample)
                pos_candidates = [idx for idx in cls_indices if idx != anchor_idx]
                if len(pos_candidates) > 0:
                    pos_idx = np.random.choice(pos_candidates)
                    positive = X[pos_idx]
                else:
                    positive = anchor  # fallback
                
                # Negative (different class)
                neg_cls = np.random.choice(other_classes)
                neg_indices = class_indices[neg_cls]
                neg_idx = np.random.choice(neg_indices)
                negative = X[neg_idx]
                
                triplets.append([anchor, positive, negative])
        
        return np.array(triplets)
    
    def __len__(self):
        return len(self.triplets)
    
    def __getitem__(self, idx):
        triplet = self.triplets[idx]
        
        # Reshape to square image based on input size
        if self.input_size is not None:
            size = self.input_size
            anchor = torch.FloatTensor(triplet[0]).view(1, size, size)
            positive = torch.FloatTensor(triplet[1]).view(1, size, size)
            negative = torch.FloatTensor(triplet[2]).view(1, size, size)
        else:
            # Auto-calculate size
            num_features = len(triplet[0])
            size = math.isqrt(num_features)
            if size * size < num_features:
                size += 1
            anchor = torch.FloatTensor(triplet[0]).view(1, size, size)
            positive = torch.FloatTensor(triplet[1]).view(1, size, size)
            negative = torch.FloatTensor(triplet[2]).view(1, size, size)
            
        return anchor, positive, negative

# ==================== 5. COMPLETE DFE MODEL ====================
class DFEComplete(nn.Module):
    def __init__(self, input_channels=1, embedding_dim=256, input_features=81):
        super(DFEComplete, self).__init__()
        self.backbone = DFEBackbone(input_channels, embedding_dim, input_features)
        self.feature_compressor = FeatureCompressor()
        
    def forward(self, x, return_features=False):
        if return_features:
            embedding, features = self.backbone(x, return_features=True)
            return embedding, features
        else:
            embedding = self.backbone(x)
            return embedding



# ==================== 6. DFE TRAINING FRAMEWORK ====================
class DFE:
    def __init__(self, embedding_dim=256, alpha1=0.5, alpha2=2.0, eta=0.5,
                 batch_size=32, learning_rate=0.001, epochs=200, lambda_weight=1.0):
        
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        logger.info(f"Using device: {self.device}")
        
        # Exact paper parameters
        self.embedding_dim = embedding_dim
        self.alpha1 = alpha1
        self.alpha2 = alpha2
        self.eta = eta
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.lambda_weight = lambda_weight  # λ

        # Exact beta values from paper page 8
        self.beta_values = [0.01, 0.02, 0.03, 0.04, 0.05, 0.05]
        
        # Dynamic attributes
        self.input_features = None
        self.input_size = None
        
        self.model = None
        self.template_library = {}
        self.label_encoder = LabelEncoder()
        self.scaler = StandardScaler()

    def diagnose_problem(self, X_train, y_train):
        """تشخیص دقیق مشکل"""
        
        print("="*50)
        print("DIAGNOSIS")
        print("="*50)
        
        # 1. Data distribution
        unique, counts = np.unique(y_train, return_counts=True)
        print(f"Classes: {unique}")
        print(f"Samples per class: {counts}")
        print(f"Min samples: {counts.min()}, Max: {counts.max()}")
        
        # 2. Feature statistics
        print(f"Feature range: [{X_train.min():.3f}, {X_train.max():.3f}]")
        print(f"Feature std: {X_train.std():.3f}")
        
        # 3. Triplet generation check
        triplet_dataset = TripletDataset(X_train, y_train, samples_per_class=10)
        print(f"Generated triplets: {len(triplet_dataset)}")
        
        # 4. Model capacity check
        total_params = sum(p.numel() for p in self.model.parameters())
        data_size = len(X_train)
        ratio = total_params / data_size
        print(f"Parameters/Data ratio: {ratio:.2f}")
        if ratio > 10:
            print("⚠️  Model too complex for data size!")
        
        print("="*50)
        
    def prepare_data(self, csv_path, is_training=True):
        """
        Load and preprocess data exactly as in paper:
        1. Extract features
        2. Reshape to square matrix (grayscale image)
        3. Normalize to [0, 1]
        """
        logger.info(f"Loading {csv_path}")
        df = pd.read_csv(csv_path)
        
        X = df.iloc[:, :-1].values
        y = df.iloc[:, -1].values
        
        # Store number of features for dynamic reshaping
        self.input_features = X.shape[1]
        self.input_size = self._calculate_input_size(self.input_features)
        
        logger.info(f"Dataset shape: {X.shape}, Classes: {len(np.unique(y))}")
        logger.info(f"Input features: {self.input_features}, Input size: {self.input_size}x{self.input_size}")
        
        # Pad or truncate features to fit square matrix
        X = self._reshape_features(X)
        
        if is_training:
            y_encoded = self.label_encoder.fit_transform(y)
            X_scaled = self.scaler.fit_transform(X)
        else:
            y_encoded = self.label_encoder.transform(y)
            X_scaled = self.scaler.transform(X)
        
        # Normalize to [0, 1] as mentioned in paper
        X_normalized = (X_scaled - X_scaled.min()) / (X_scaled.max() - X_scaled.min() + 1e-8)
        
        return X_normalized, y_encoded
    
    def _calculate_input_size(self, num_features):
        """Calculate the closest square size for given number of features"""
        sqrt = math.isqrt(num_features)
        
        # Check if it's a perfect square
        if sqrt * sqrt == num_features:
            return sqrt
        
        # Otherwise find the closest square that can contain all features
        for size in range(sqrt + 1, 0, -1):
            if size * size >= num_features:
                return size
        
        return sqrt + 1  # Fallback
    
    def _reshape_features(self, X):
        """Reshape features to square matrix, padding with zeros if needed"""
        n_samples = X.shape[0]
        target_size = self.input_size * self.input_size
        
        if X.shape[1] < target_size:
            # Pad with zeros
            padding = np.zeros((n_samples, target_size - X.shape[1]))
            X_reshaped = np.concatenate([X, padding], axis=1)
        elif X.shape[1] > target_size:
            # Truncate
            X_reshaped = X[:, :target_size]
        else:
            X_reshaped = X
        
        return X_reshaped
    
    def train(self, train_csv_path):
        """Train DFE model exactly as in paper"""
        logger.info("="*60)
        logger.info("STARTING DFE TRAINING (100% Paper Implementation)")
        logger.info("="*60)
        
        # Load data
        X_train, y_train = self.prepare_data(train_csv_path, is_training=True)
        
        # Create triplet dataset with dynamic size
        triplet_dataset = TripletDataset(X_train, y_train, 
                                       samples_per_class=4000, 
                                       input_size=self.input_size)
        triplet_loader = DataLoader(triplet_dataset, batch_size=self.batch_size, 
                                  shuffle=True, drop_last=True, num_workers=0)
        
        logger.info(f"Created {len(triplet_dataset)} triplets")
        logger.info(f"Input shape for network: (batch, 1, {self.input_size}, {self.input_size})")
        
        # Initialize model with dynamic architecture
        self.model = DFEComplete(
            input_channels=1, 
            embedding_dim=self.embedding_dim, 
            input_features=self.input_features
        ).to(self.device)
        
        # Count parameters
        total_params = sum(p.numel() for p in self.model.parameters())
        logger.info(f"Model parameters: {total_params:,}")
        logger.info(f"Architecture: EXACT Figure 2 - 13 conv layers with residual connections")
        logger.info(f"Input dimensions: {self.input_size}x{self.input_size}")
        
        # Loss functions (Equations 12 and 13)
        triplet_loss_fn = ModifiedTripletLoss(alpha1=self.alpha1, alpha2=self.alpha2, eta=self.eta)
        feature_compressor = FeatureCompressor(beta_values=self.beta_values)
        
        # Optimizer: SGD with constant LR as in paper
        optimizer = torch.optim.SGD(self.model.parameters(), lr=self.learning_rate, 
                                  momentum=0.9, weight_decay=1e-4)
        
        # Learning rate scheduler
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)
        
        logger.info(f"Training for {self.epochs} epochs")
        logger.info(f"Batch size: {self.batch_size}")
        logger.info(f"Learning rate: {self.learning_rate}")
        logger.info(f"α1={self.alpha1}, α2={self.alpha2}, η={self.eta}")
        logger.info(f"β={self.beta_values}, λ={self.lambda_weight}")
        
        best_loss = float('inf')
        patience_counter = 0
        patience = 30  # Early stopping patience
        
        for epoch in range(self.epochs):
            self.model.train()
            epoch_loss = 0
            epoch_triplet_loss = 0
            epoch_compression_loss = 0
            num_batches = 0
            
            for batch_idx, (anchors, positives, negatives) in enumerate(triplet_loader):
                # Move to device
                anchors = anchors.to(self.device)
                positives = positives.to(self.device) 
                negatives = negatives.to(self.device)
                
                optimizer.zero_grad()
                
                try:
                    # Forward pass with features for compression
                    anchor_emb, anchor_features = self.model(anchors, return_features=True)
                    positive_emb = self.model(positives)
                    negative_emb = self.model(negatives)
                    
                    # Triplet loss (L_NTC from Equation 12)
                    triplet_loss = triplet_loss_fn(anchor_emb, positive_emb, negative_emb)
                    
                    # Feature compression loss (L_f from Equation 8)
                    compression_loss = feature_compressor.compute_compression_loss(anchor_features)
                    
                    # Total loss (Equation 13: L = L_f + λ * L_NTC)
                    total_loss = compression_loss + self.lambda_weight * triplet_loss
                    
                    # Backward pass
                    total_loss.backward()
                    
                    # Gradient clipping for stability
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                    
                    optimizer.step()
                    
                    epoch_loss += total_loss.item()
                    epoch_triplet_loss += triplet_loss.item()
                    epoch_compression_loss += compression_loss.item()
                    num_batches += 1
                    
                except Exception as e:
                    logger.warning(f"Error in batch {batch_idx}: {e}")
                    continue
            
            # Update learning rate
            scheduler.step()
            
            if num_batches > 0:
                avg_loss = epoch_loss / num_batches
                avg_triplet = epoch_triplet_loss / num_batches
                avg_compression = epoch_compression_loss / num_batches
                
                # Log progress
                if epoch % 10 == 0 or epoch < 10:
                    current_lr = optimizer.param_groups[0]['lr']
                    logger.info(f"Epoch [{epoch+1:3d}/{self.epochs}] "
                              f"Loss: {avg_loss:.4f} "
                              f"(Triplet: {avg_triplet:.4f}, "
                              f"Compression: {avg_compression:.4f}) "
                              f"LR: {current_lr:.6f}")
                
                # Early stopping with patience
                if avg_loss < best_loss:
                    best_loss = avg_loss
                    patience_counter = 0
                    
                    # Save best model
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': self.model.state_dict(),
                        'loss': avg_loss,
                        'input_features': self.input_features,
                        'input_size': self.input_size,
                        'alpha1': self.alpha1,
                        'alpha2': self.alpha2,
                        'beta_values': self.beta_values
                    }, 'dfe_best_model.pth')
                else:
                    patience_counter += 1
                    
                if patience_counter >= patience and epoch > 50:
                    logger.info(f"Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
                    break
        self.diagnose_problem(X_train, y_train)

        logger.info("Training completed!")
        
        # Load best model
        if os.path.exists('dfe_best_model.pth'):
            checkpoint = torch.load('dfe_best_model.pth', map_location=self.device)
            self.model.load_state_dict(checkpoint['model_state_dict'])
            logger.info(f"Loaded best model from epoch {checkpoint['epoch']+1} with loss {checkpoint['loss']:.4f}")
        
        # Build template library with 20 samples per class (exactly as in paper)
        self._build_template_library(X_train, y_train)
    
    def _build_template_library(self, X, y):
        """
        Build template library with 20 samples per class as mentioned in paper page 8
        """
        logger.info("Building template library...")
        self.model.eval()
        
        unique_classes = np.unique(y)
        
        with torch.no_grad():
            for cls in unique_classes:
                class_indices = np.where(y == cls)[0]
                
                # Take up to 20 samples per class (exact paper specification)
                n_samples = min(20, len(class_indices))
                selected_indices = np.random.choice(class_indices, n_samples, replace=False)
                
                # Get embeddings
                class_samples = X[selected_indices]
                # Reshape to square images
                class_samples_reshaped = class_samples.reshape(-1, 1, self.input_size, self.input_size)
                class_samples_tensor = torch.FloatTensor(class_samples_reshaped).to(self.device)
                
                embeddings = self.model(class_samples_tensor)
                
                # Average embedding as template (as mentioned in paper)
                template = torch.mean(embeddings, dim=0).cpu().numpy()
                self.template_library[cls] = template
        
        logger.info(f"Template library created with {len(self.template_library)} classes")
    
    def evaluate(self, test_csv_path):
        """
        Evaluate model using template matching with L2-norm distance
        Exactly as described in paper Section III.C
        """
        logger.info("="*60)
        logger.info("EVALUATION (Template Matching)")
        logger.info("="*60)
        
        X_test, y_test = self.prepare_data(test_csv_path, is_training=False)
        
        # Predict using template matching
        y_pred = self._predict_with_template_matching(X_test)
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        
        logger.info(f"Accuracy:  {accuracy:.4f}")
        logger.info(f"Precision: {precision:.4f}")
        logger.info(f"Recall:    {recall:.4f}")
        logger.info(f"F1-Score:  {f1:.4f}")
        
        # Confusion matrix (optional)
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(y_test, y_pred)
        logger.info(f"Confusion matrix shape: {cm.shape}")
        
        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'confusion_matrix': cm
        }
    
    def _predict_with_template_matching(self, X, k=1):
        """
        Template matching exactly as in paper Equation 14:
        y = arg max_c_j Σ_{x_i ∈ N_k(x)} I(y_i = c_j)
        
        For k=1, this is simply nearest template
        """
        self.model.eval()
        predictions = []
        
        batch_size = 32
        with torch.no_grad():
            for i in range(0, len(X), batch_size):
                batch_X = X[i:i + batch_size]
                # Reshape to square images
                batch_X_reshaped = batch_X.reshape(-1, 1, self.input_size, self.input_size)
                X_tensor = torch.FloatTensor(batch_X_reshaped).to(self.device)
                
                embeddings = self.model(X_tensor).cpu().numpy()
                
                for emb in embeddings:
                    # Calculate L2-norm distance to all templates
                    min_distance = float('inf')
                    predicted_class = -1
                    
                    for cls, template in self.template_library.items():
                        distance = np.linalg.norm(emb - template)  # L2-norm
                        
                        if distance < min_distance:
                            min_distance = distance
                            predicted_class = cls
                    
                    predictions.append(predicted_class)
        
        return np.array(predictions)
    
    def add_new_traffic_type(self, new_samples, new_class_name):
        """
        Add new traffic type to template library without retraining
        This demonstrates the key advantage of DFE mentioned in paper
        """
        logger.info(f"Adding new traffic type: {new_class_name}")
        
        # Encode new class
        new_class_encoded = len(self.template_library)  # Simple encoding
        
        # Preprocess new samples
        new_samples_scaled = self.scaler.transform(new_samples)
        new_samples_normalized = (new_samples_scaled - new_samples_scaled.min()) / \
                               (new_samples_scaled.max() - new_samples_scaled.min() + 1e-8)
        
        # Reshape to square
        new_samples_reshaped = self._reshape_features(new_samples_normalized)
        
        # Get embeddings
        self.model.eval()
        with torch.no_grad():
            samples_reshaped = new_samples_reshaped.reshape(-1, 1, self.input_size, self.input_size)
            samples_tensor = torch.FloatTensor(samples_reshaped).to(self.device)
            embeddings = self.model(samples_tensor)
            
            # Create template
            template = torch.mean(embeddings, dim=0).cpu().numpy()
            self.template_library[new_class_encoded] = template
        
        logger.info(f"New traffic type added to template library")
        return new_class_encoded
    
    def save_model(self, path='dfe_complete_model.pth'):
        """Save the complete model with all components"""
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'template_library': self.template_library,
            'label_encoder': self.label_encoder,
            'scaler': self.scaler,
            'input_features': self.input_features,
            'input_size': self.input_size,
            'parameters': {
                'embedding_dim': self.embedding_dim,
                'alpha1': self.alpha1,
                'alpha2': self.alpha2,
                'eta': self.eta,
                'beta_values': self.beta_values,
                'lambda_weight': self.lambda_weight
            }
        }, path)
        logger.info(f"Model saved to {path}")
    
    def load_model(self, path='dfe_complete_model.pth'):
        """Load a trained model"""
        checkpoint = torch.load(path, map_location=self.device)
        
        # Restore parameters
        self.input_features = checkpoint['input_features']
        self.input_size = checkpoint['input_size']
        
        # Initialize model
        self.model = DFEComplete(
            input_channels=1,
            embedding_dim=self.embedding_dim,
            input_features=self.input_features
        ).to(self.device)
        
        # Load weights
        self.model.load_state_dict(checkpoint['model_state_dict'])
        
        # Restore other components
        self.template_library = checkpoint['template_library']
        self.label_encoder = checkpoint['label_encoder']
        self.scaler = checkpoint['scaler']
        
        logger.info(f"Model loaded from {path}")
        logger.info(f"Input features: {self.input_features}, Input size: {self.input_size}x{self.input_size}")

# ==================== 7. MAIN FUNCTION ====================
def main():
    print("="*80)
    print("DEEP FLOW EMBEDDING (DFE) - 100% EXACT PAPER IMPLEMENTATION")
    print("Based on: Wang et al., IEEE Trans. Network Science and Engineering, 2025")
    print("="*80)
    
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    
    # Initialize DFE with EXACT paper parameters (page 8)
    dfe = DFE(
        embedding_dim=256,    # Paper specification
        alpha1=0.5,           # Margin for inter-class constraint (paper page 8)
        alpha2=2.0,           # Margin for intra-class constraint (paper page 8)
        eta=0.5,              # Weight coefficient η in Equation 12
        batch_size=32,        # Paper page 8
        learning_rate=0.005,  # Paper page 8
        epochs=20,           # Paper page 8
        lambda_weight=1.0     # λ in Equation 13
    )
    
    # Dataset paths
    # Use your CSV files
    train_csv = "../DATASETS/CDR-MLC/level_2.csv"
    test_csv = "../DATASETS/CDR-MLC/level_3.csv"
    
    try:
        # Train the model
        start_time = time.time()
        dfe.train(train_csv)
        training_time = time.time()
        training_time = time.time() - start_time
        
        logger.info(f"Training completed in {training_time:.2f} seconds")
        
        # Evaluate the model
        start_time = time.time()
        results = dfe.evaluate(test_csv)
        inference_time = time.time() - start_time
        
        logger.info(f"Evaluation completed in {inference_time:.2f} seconds")
        
        # Save the final model with all components
        dfe.save_model('dfe_complete_model.pth')
        
        # Display final results
        print("\n" + "="*60)
        print("FINAL RESULTS")
        print("="*60)
        print(f"Input Features: {dfe.input_features}")
        print(f"Input Size: {dfe.input_size}x{dfe.input_size}")
        print(f"Accuracy:  {results['accuracy']:.4f}")
        print(f"Precision: {results['precision']:.4f}")
        print(f"Recall:    {results['recall']:.4f}")
        print(f"F1-Score:  {results['f1_score']:.4f}")
        print(f"Training Time: {training_time:.2f} seconds")
        print(f"Inference Time: {inference_time:.2f} seconds")
        print(f"Model saved to: dfe_complete_model.pth")
        print("="*60)
        

    except FileNotFoundError as e:
        logger.error(f"File not found: {e}")
        print("\nPlease check the CSV file paths:")
        print(f"Train CSV: {train_csv}")
        print(f"Test CSV: {test_csv}")
        print("\nTo use this code:")
        print("1. Update the CSV file paths")
        print("2. Ensure CSV has features in columns and label in last column")
        print("3. The code automatically handles any number of features")
        
    except Exception as e:
        logger.error(f"Error during execution: {e}")
        import traceback


# ==================== 9. RUN MAIN ====================
if __name__ == "__main__":
    # Run the main function
    main()
    
    # Optional: Test with different feature sizes
    # test_with_different_features()

2026-02-15 11:36:50,972 - INFO - Using device: cuda
2026-02-15 11:36:50,972 - INFO - ============================================================
2026-02-15 11:36:50,972 - INFO - STARTING DFE TRAINING (100% Paper Implementation)
2026-02-15 11:36:50,972 - INFO - ============================================================
2026-02-15 11:36:50,972 - INFO - Loading ../DATASETS/CDR-MLC/level_2.csv


DEEP FLOW EMBEDDING (DFE) - 100% EXACT PAPER IMPLEMENTATION
Based on: Wang et al., IEEE Trans. Network Science and Engineering, 2025


2026-02-15 11:36:51,375 - INFO - Dataset shape: (201165, 36), Classes: 5
2026-02-15 11:36:51,376 - INFO - Input features: 36, Input size: 6x6
2026-02-15 11:37:55,801 - INFO - Created 20000 triplets
2026-02-15 11:37:55,801 - INFO - Input shape for network: (batch, 1, 6, 6)
2026-02-15 11:37:55,801 - INFO - Input features: 36, Input size: 6x6
2026-02-15 11:37:55,892 - INFO - Model parameters: 2,869,952
2026-02-15 11:37:55,892 - INFO - Architecture: EXACT Figure 2 - 13 conv layers with residual connections
2026-02-15 11:37:55,892 - INFO - Input dimensions: 6x6
2026-02-15 11:38:13,835 - INFO - Training for 20 epochs
2026-02-15 11:38:13,837 - INFO - Batch size: 32
2026-02-15 11:38:13,837 - INFO - Learning rate: 0.005
2026-02-15 11:38:13,837 - INFO - α1=0.5, α2=2.0, η=0.5
2026-02-15 11:38:13,838 - INFO - β=[0.01, 0.02, 0.03, 0.04, 0.05, 0.05], λ=1.0
2026-02-15 11:38:44,980 - INFO - Epoch [  1/20] Loss: 1.5920 (Triplet: 0.5250, Compression: 1.0669) LR: 0.005000
2026-02-15 11:39:15,557 - INFO -

DIAGNOSIS
Classes: [0 1 2 3 4]
Samples per class: [114530  42437  14730   9501  19967]
Min samples: 9501, Max: 114530
Feature range: [0.000, 1.000]
Feature std: 0.003
Generated triplets: 50
Parameters/Data ratio: 14.27
⚠️  Model too complex for data size!


2026-02-15 11:47:48,411 - INFO - Loaded best model from epoch 5 with loss 1.5475
2026-02-15 11:47:48,411 - INFO - Building template library...
2026-02-15 11:47:48,442 - INFO - Template library created with 5 classes
2026-02-15 11:47:48,445 - INFO - Training completed in 657.47 seconds
2026-02-15 11:47:48,445 - INFO - ============================================================
2026-02-15 11:47:48,445 - INFO - EVALUATION (Template Matching)
2026-02-15 11:47:48,446 - INFO - ============================================================
2026-02-15 11:47:48,446 - INFO - Loading ../DATASETS/CDR-MLC/level_3.csv
2026-02-15 11:47:48,565 - INFO - Dataset shape: (60482, 36), Classes: 5
2026-02-15 11:47:48,565 - INFO - Input features: 36, Input size: 6x6
2026-02-15 11:47:53,193 - INFO - Accuracy:  0.1594
2026-02-15 11:47:53,194 - INFO - Precision: 0.0254
2026-02-15 11:47:53,194 - INFO - Recall:    0.1594
2026-02-15 11:47:53,194 - INFO - F1-Score:  0.0438
2026-02-15 11:47:53,198 - INFO - Confusion m


FINAL RESULTS
Input Features: 36
Input Size: 6x6
Accuracy:  0.1594
Precision: 0.0254
Recall:    0.1594
F1-Score:  0.0438
Training Time: 657.47 seconds
Inference Time: 4.75 seconds
Model saved to: dfe_complete_model.pth
